# Preprocesado de texto: traduccion
Traduce al ingles solo las descripciones marcadas como malayo o chino (mismas reglas que `txt_stats` de texto-v2:
`txt_frac_chino > 0.3` o `txt_frac_malayo > 0.15`) con NLLB-200 y vuelve a correr el modelo de emociones del
preprocesado original (`j-hartmann/emotion-english-distilroberta-base`) sobre el texto traducido.
Salida: `emo_trad_{train,test}.parquet` (solo las mascotas traducidas, mismas 7 columnas que el original) y
`trad_{train,test}.parquet` (texto original y traducido, para revisar).

In [ ]:
# Kaggle monta los inputs en otra carpeta cuando el notebook se crea por API: armar ../input/<nombre>
import os, glob
def _buscar(nombre, archivo):
    for prof in range(0, 5):
        for d in glob.glob("/kaggle/input/" + "*/" * prof):
            d = d.rstrip("/")
            if os.path.basename(d).endswith(nombre) and os.path.exists(os.path.join(d, archivo)):
                return d
    raise FileNotFoundError(nombre)
_inputs = {"petfinder-adoption-prediction": "train/train.csv", "examen-lab-ii-preprocesado-texto": "train.parquet"}
if not all(os.path.exists(f"../input/{n}/{a}") for n, a in _inputs.items()):
    os.makedirs("/tmp/lab/input", exist_ok=True)
    os.makedirs("/tmp/lab/work", exist_ok=True)
    for n, a in _inputs.items():
        if not os.path.exists(f"/tmp/lab/input/{n}"):
            os.symlink(_buscar(n, a), f"/tmp/lab/input/{n}")
    os.chdir("/tmp/lab/work")
D = "../input/petfinder-adoption-prediction"
OUT = "/kaggle/working" if os.path.exists("/kaggle/working") else "."

In [ ]:
import re
import numpy as np
import pandas as pd
train = pd.read_csv(f"{D}/train/train.csv").set_index("PetID")
test = pd.read_csv(f"{D}/test/test.csv").set_index("PetID")
ids_train, ids_test = train.index, test.index
# el preprocesado original usa fillna("empty") sin strip: igual aca, para que lo no traducido de lo mismo
desc = pd.concat([train["Description"], test["Description"]]).fillna("empty").astype(str)
assert desc.index.is_unique

# marcas de idioma: mismas definiciones que txt_stats (preprocesado texto-v2)
MALAYO = {"dan", "yang", "untuk", "ini", "itu", "boleh", "tidak", "saya", "dia", "ada", "sangat", "anak",
          "kucing", "anjing", "dengan", "nak", "jantan", "betina", "sudah", "sihat", "comel", "manja", "akan",
          "lagi", "sila", "hubungi", "percuma", "mahu", "kami", "dari", "pada", "dalam", "tak", "dah"}
def frac_chino(t):
    t = t.strip()
    return len(re.findall(r"[\u4e00-\u9fff]", t)) / len(t) if t else 0.0
def frac_malayo(t):
    p = re.findall(r"[A-Za-z]+", t.strip())
    return sum(w.lower() in MALAYO for w in p) / len(p) if p else 0.0
fc, fm = desc.map(frac_chino), desc.map(frac_malayo)
idioma = pd.Series("", index=desc.index)
idioma[fm > 0.15] = "zsm_Latn"
idioma[fc > 0.3] = "zho_Hans"   # chino gana si hay las dos marcas
marcadas = idioma[idioma != ""]
print("a traducir: train", marcadas.index.isin(ids_train).sum(), "test", marcadas.index.isin(ids_test).sum())
print(marcadas.value_counts())

In [ ]:
import torch
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:
    (torch.ones(4, 4, device=dev) @ torch.ones(4, 4, device=dev)).sum().item()
except Exception as e:
    print("cuda no anda, sigo en cpu:", e)
    dev = "cpu"
print("torch", torch.__version__, dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")

In [ ]:
# traduccion con NLLB-200: por pedazos de oraciones (el modelo traduce bien hasta ~200 tokens)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
MT = "facebook/nllb-200-distilled-600M"
mt_tok = AutoTokenizer.from_pretrained(MT)
mt = AutoModelForSeq2SeqLM.from_pretrained(MT).to(dev).eval()
if dev == "cuda":
    mt = mt.half()
ENG = mt_tok.convert_tokens_to_ids("eng_Latn")

def pedazos(t, max_tok=150):
    oraciones = [o for o in re.split(r"(?<=[.!?\u3002\uff01\uff1f])\s*|\n+", t) if o and o.strip()]
    out, cur = [], ""
    for o in oraciones:
        cand = (cur + " " + o).strip()
        if cur and len(mt_tok(cand)["input_ids"]) > max_tok:
            out.append(cur)
            cur = o.strip()
        else:
            cur = cand
    if cur:
        out.append(cur)
    return out or [t]

traduccion = {}
with torch.no_grad():
    for lang in marcadas.unique():
        ids = marcadas.index[marcadas == lang]
        mt_tok.src_lang = lang
        filas = [(i, p) for i in ids for p in pedazos(desc[i])]
        res = []
        for k in range(0, len(filas), 32):
            lote = [p for _, p in filas[k:k + 32]]
            enc = mt_tok(lote, return_tensors="pt", padding=True, truncation=True, max_length=256).to(dev)
            gen = mt.generate(**enc, forced_bos_token_id=ENG, max_new_tokens=300, num_beams=2)
            res += mt_tok.batch_decode(gen, skip_special_tokens=True)
        por_pet = {}
        for (i, _), r in zip(filas, res):
            por_pet.setdefault(i, []).append(r)
        traduccion.update({i: " ".join(v) for i, v in por_pet.items()})
        print(lang, len(ids), "mascotas,", len(filas), "pedazos", flush=True)
trad = pd.DataFrame({"idioma": marcadas, "original": desc[marcadas.index], "traduccion": pd.Series(traduccion)})
assert trad["traduccion"].notna().all()
for i in trad.index[:3].tolist() + trad.index[trad.idioma == "zho_Hans"][:3].tolist():
    print("==", trad.loc[i, "idioma"], "|", trad.loc[i, "original"][:200].replace("\n", " "))
    print("  ->", trad.loc[i, "traduccion"][:200])
del mt
torch.cuda.empty_cache() if dev == "cuda" else None

In [ ]:
# mismo modelo de emociones que el preprocesado original, sobre el texto traducido
from transformers import pipeline
EMO = "j-hartmann/emotion-english-distilroberta-base"
clf = pipeline("text-classification", model=EMO, top_k=None, device=0 if dev == "cuda" else -1)
def emociones(textos, idx):
    r = clf(list(textos), max_length=512, truncation=True)
    return pd.DataFrame([{d["label"]: d["score"] for d in x} for x in r], index=idx)

orig = pd.concat([pd.read_parquet("../input/examen-lab-ii-preprocesado-texto/train.parquet"),
                  pd.read_parquet("../input/examen-lab-ii-preprocesado-texto/test.parquet")])
# control: con el texto original tiene que dar lo mismo que el preprocesado original
control = emociones(trad["original"], trad.index)[orig.columns]
print("control: max |dif| con el original =", float((control - orig.loc[trad.index]).abs().max().max()))
emo = emociones(trad["traduccion"], trad.index)[orig.columns]
print("cambio medio por emocion (traducido - original):")
print((emo - orig.loc[trad.index]).mean().round(3).to_string())
for parte, ids in (("train", ids_train), ("test", ids_test)):
    emo[emo.index.isin(ids)].to_parquet(f"{OUT}/emo_trad_{parte}.parquet")
    trad[trad.index.isin(ids)].to_parquet(f"{OUT}/trad_{parte}.parquet")
print(emo.shape, list(emo.columns))